# Week 06 — Classification Algorithms
### Comparative Multi-Model Benchmark on Loan Default Prediction
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

df = pd.read_csv("datasets/loan_default_data.csv")
print(f"Dataset shape: {df.shape} | Default rate: {df['Default'].mean()*100:.1f}%")
df.head()


## 1. Feature Preprocessing & Partitioning


In [ ]:
X = df.drop(columns=["Default"])
y = df["Default"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 2. Multi-Algorithm Benchmark Execution


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

results = []
for name, clf in models.items():
    # Use scaled data for all for consistency
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    probs = clf.predict_proba(X_test_scaled)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-Score": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs)
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
results_df


## 3. Confusion Matrix & ROC Curve Comparison


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
cm = confusion_matrix(y_test, best_model.predict(X_test_scaled))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Non-Default", "Default"]).plot(ax=ax, cmap="Blues")
plt.title(f"Confusion Matrix: {best_model_name}", fontsize=13, fontweight="bold")
plt.show()


## 4. Conclusion & Model Selection
*(Interns: Explain which model should be deployed in production and why)*
